# Review Full Dataset Data Sources

Use this notebook to formally inspect the external data sources for the full `fake_mantis_invest` universe. The goal here is not only to fetch the data, but to dereference URLs and filing paths, extract preview text, and judge how useful each source will be for the long-term product.


## Scope

- Build the full ticker universe from the fake Mantis dataset.
- Fetch official SEC company mappings and filing metadata.
- Build full-document URLs for latest 10-K / 10-Q filings.
- Inflate a sample of those filing URLs into extracted text previews.
- Fetch macro context from FRED.
- Fetch company profile enrichment from FMP.
- Fetch broad news metadata from GDELT.
- Inflate a sample of news URLs into extracted text previews.
- Create a source-review table describing what each source is useful for.

### Helpful Short Forms

- `SEC` = **U.S. Securities and Exchange Commission**
- `CIK` = **Central Index Key**, the SEC company identifier
- `FRED` = **Federal Reserve Economic Data**
- `GDELT` = **Global Database of Events, Language, and Tone**
- `FMP` = **Financial Modeling Prep**
- `10-K` = annual filing
- `10-Q` = quarterly filing
- `8-K` = current event filing
- `MD&A` = **Management's Discussion and Analysis**


In [ ]:
from __future__ import annotations

import gzip
import json
import os
import re
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
REVIEW_DIR = INTERIM_DIR / 'source_reviews'
load_dotenv(ROOT / '.env')

for path in [
    EXTERNAL_DIR / 'sec',
    EXTERNAL_DIR / 'fred',
    EXTERNAL_DIR / 'gdelt',
    EXTERNAL_DIR / 'fmp',
    REVIEW_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

SEC_HEADERS = {
    'User-Agent': 'portfolio-analyzer research contact@example.com',
    'Accept-Encoding': 'gzip, deflate',
}

MAX_SEC_DOCS_TO_INFLATE = 10
MAX_GDELT_ARTICLES_PER_TICKER = 3
MAX_NEWS_URLS_TO_INFLATE = 10

def read_response_bytes(response) -> bytes:
    payload = response.read()
    encoding = (response.headers.get('Content-Encoding') or '').lower()
    if 'gzip' in encoding:
        payload = gzip.decompress(payload)
    return payload

def fetch_url(url: str, headers: dict[str, str] | None = None, *, retries: int = 4, base_delay: float = 1.5) -> bytes:
    merged_headers = headers or {}
    last_error = None
    for attempt in range(retries):
        request = Request(url, headers=merged_headers)
        try:
            with urlopen(request) as response:
                return read_response_bytes(response)
        except HTTPError as error:
            last_error = error
            if error.code != 429 or attempt == retries - 1:
                raise
            retry_after = error.headers.get('Retry-After')
            delay = float(retry_after) if retry_after else base_delay * (2 ** attempt)
            print(f'Rate limited for {url}. Sleeping {delay:.1f}s before retry {attempt + 2}/{retries}.')
            time.sleep(delay)
    raise last_error if last_error else RuntimeError('Request failed without a captured HTTPError')

def fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:
    return json.loads(fetch_url(url, headers=headers).decode('utf-8'))

def fetch_text(url: str, headers: dict[str, str] | None = None) -> str:
    return fetch_url(url, headers=headers).decode('utf-8', errors='ignore')

def html_to_text(html: str) -> str:
    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)
    no_tags = re.sub(r'<[^>]+>', ' ', no_script)
    return re.sub(r'\s+', ' ', no_tags).strip()

def clean_filing_text(text: str) -> str:
    clean = text
    clean = re.sub(r'\b(?:us-gaap|dei|xbrli|xbrldi|srt|link|iso4217):[A-Za-z0-9_]+\b', ' ', clean)
    clean = re.sub(r'https?://\S+', ' ', clean)
    clean = re.sub(r'\b\d{10}[- ]?\d{2}[- ]?\d{6}\b', ' ', clean)
    clean = re.sub(r'\bCIK\b|\bExact name of registrant as specified in its charter\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\bPage\s+\d+\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\bTable of Contents\b', ' ', clean, flags=re.I)
    clean = re.sub(r'[_•·]+', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean


def clean_article_text(text: str) -> str:
    clean = text
    clean = re.sub(r'https?://\S+', ' ', clean)
    clean = re.sub(r'\b(?:cookie|privacy policy|terms of service|advertisement|subscribe|sign up|newsletter)\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

def simple_chunk(text: str, chunk_size: int = 1800) -> list[str]:
    clean = re.sub(r'\s+', ' ', text).strip()
    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]

def find_item_heading(text: str, candidates: list[str]) -> int | None:
    lowered = text.lower()
    matches = []
    for candidate in candidates:
        idx = lowered.find(candidate.lower())
        if idx >= 0:
            matches.append(idx)
    return min(matches) if matches else None

def extract_section_preview(text: str, start_candidates: list[str], *, preview_chars: int = 2200) -> str:
    start = find_item_heading(text, start_candidates)
    if start is None:
        return ''
    next_match = re.search(r'item\s+\d+[a-z]?\.?\s', text[start + 20:], flags=re.I)
    end = start + 20 + next_match.start() if next_match else min(len(text), start + preview_chars)
    snippet = text[start:end]
    return snippet[:preview_chars].strip()

def get_filing_section_previews(clean_text: str, form_type: str) -> dict[str, str]:
    if form_type == "10-K":
        risk = extract_section_preview(clean_text, ["item 1a. risk factors", "item 1a risk factors", "risk factors"])
        mda = extract_section_preview(clean_text, ["item 7. management\'s discussion and analysis", "item 7 management\'s discussion and analysis", "management\'s discussion and analysis"])
        business = extract_section_preview(clean_text, ["item 1. business", "item 1 business"])
    else:
        risk = extract_section_preview(clean_text, ["item 1a. risk factors", "item 1a risk factors", "risk factors"])
        mda = extract_section_preview(clean_text, ["item 2. management\'s discussion and analysis", "item 2 management\'s discussion and analysis", "management\'s discussion and analysis"])
        business = extract_section_preview(clean_text, ["business"])
    return {
        'risk_factors_preview': risk,
        'mda_preview': mda,
        'business_preview': business,
    }

def build_primary_doc_url(cik: str, accession_number: str, primary_document: str) -> str:
    accession = accession_number.replace('-', '')
    return f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary_document}'

transactions = pd.read_csv(RAW_DIR / 'fake_mantis_invest.csv')
ticker_universe = sorted({
    str(value).strip()
    for value in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist()
    if str(value).strip()
})
len(ticker_universe), ticker_universe[:10]

## 1. Full Ticker Universe

Build the full stock / ETF universe from the fake Mantis dataset so every later source review is grounded in the actual portfolio universe.


In [ ]:
ticker_overview = pd.DataFrame({'ticker': ticker_universe})
ticker_overview['is_blank'] = ticker_overview['ticker'].eq('')
ticker_overview.head(20)


## 2. SEC Company Master + Filing Metadata For Full Universe

This creates the official SEC mapping from ticker to CIK, then collects the recent filing metadata for every mapped company.


In [ ]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'
company_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)
company_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})
company_master['ticker'] = company_master['ticker'].astype(str).str.upper()
company_master['cik'] = company_master['cik'].astype(str).str.zfill(10)
company_master = company_master[company_master['ticker'].isin(ticker_universe)].copy()
company_master.to_csv(EXTERNAL_DIR / 'sec' / 'company_master_full_universe.csv', index=False)
company_master.head(20)


In [ ]:
filing_frames = []
for row in company_master.itertuples(index=False):
    submissions_url = f'https://data.sec.gov/submissions/CIK{row.cik}.json'
    submissions = fetch_json(submissions_url, headers=SEC_HEADERS)
    recent = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))
    if recent.empty:
        continue
    recent = recent[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].copy()
    recent['ticker'] = row.ticker
    recent['company_name'] = row.company_name
    recent['cik'] = row.cik
    recent['primary_document_url'] = recent.apply(
        lambda r: build_primary_doc_url(row.cik, r['accessionNumber'], r['primaryDocument']),
        axis=1,
    )
    filing_frames.append(recent)

filings_full = pd.concat(filing_frames, ignore_index=True) if filing_frames else pd.DataFrame()
filings_full.to_csv(EXTERNAL_DIR / 'sec' / 'filings_full_universe.csv', index=False)
filings_full.head(20)


## 3. Latest 10-K / 10-Q URLs + SEC Company Facts

This step narrows the SEC filing list to the latest annual / quarterly filing per ticker and also builds both a review summary and a flattened company-facts dataset you can inspect directly.


In [ ]:
latest_reporting_filings = (
    filings_full[filings_full['form'].isin(['10-K', '10-Q'])]
    .sort_values(['ticker', 'filingDate'], ascending=[True, False])
    .drop_duplicates(subset=['ticker'], keep='first')
    .reset_index(drop=True)
)
latest_reporting_filings.to_csv(EXTERNAL_DIR / 'sec' / 'latest_reporting_filings.csv', index=False)
latest_reporting_filings[['ticker', 'form', 'filingDate', 'primaryDocument', 'primary_document_url']].head(20)


In [ ]:
company_fact_rows = []
company_fact_detail_rows = []
for row in company_master.itertuples(index=False):
    facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{row.cik}.json'
    try:
        company_facts = fetch_json(facts_url, headers=SEC_HEADERS)
        us_gaap = company_facts.get('facts', {}).get('us-gaap', {})
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': 'available',
            'fact_tag_count': len(us_gaap),
            'sample_fact_tags': ', '.join(list(us_gaap.keys())[:10]),
        })

        for fact_tag, fact_payload in us_gaap.items():
            units = fact_payload.get('units', {})
            for unit_name, observations in units.items():
                for observation in observations[:25]:
                    company_fact_detail_rows.append({
                        'ticker': row.ticker,
                        'company_name': row.company_name,
                        'cik': row.cik,
                        'fact_tag': fact_tag,
                        'unit': unit_name,
                        'value': observation.get('val'),
                        'filed_date': observation.get('filed'),
                        'fiscal_year': observation.get('fy'),
                        'fiscal_period': observation.get('fp'),
                        'frame': observation.get('frame'),
                        'form': observation.get('form'),
                        'fact_start': observation.get('start'),
                        'fact_end': observation.get('end'),
                    })
    except HTTPError as error:
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': f'http_{error.code}',
            'fact_tag_count': 0,
            'sample_fact_tags': '',
        })

company_facts_review = pd.DataFrame(company_fact_rows)
company_facts_review.to_csv(EXTERNAL_DIR / 'sec' / 'company_facts_review.csv', index=False)

company_facts_full_df = pd.DataFrame(company_fact_detail_rows)
company_facts_full_df.to_csv(EXTERNAL_DIR / 'sec' / 'company_facts_full.csv', index=False)

display(company_facts_review.head(20))
company_facts_full_df.head(20) if not company_facts_full_df.empty else pd.DataFrame()

## 4. Inflate SEC Filing URLs Into Full Text + Review

This section now keeps both review tables and a fuller extracted text dataset for the inflated SEC filing URLs.


In [ ]:
latest_reporting_filings['primary_document_url'].head(1)

In [ ]:
sec_document_inventory_rows = []
sec_document_text_rows = []
sec_cleaning_review_rows = []
sec_section_preview_rows = []

for row in latest_reporting_filings.head(MAX_SEC_DOCS_TO_INFLATE).itertuples(index=False):
    html = fetch_text(row.primary_document_url, headers=SEC_HEADERS)
    raw_text = html_to_text(html)
    clean_text = clean_filing_text(raw_text)
    raw_chunks = simple_chunk(raw_text)
    clean_chunks = simple_chunk(clean_text)
    section_previews = get_filing_section_previews(clean_text, row.form)
    selected_text = section_previews['risk_factors_preview'] or section_previews['mda_preview'] or section_previews['business_preview'] or (clean_chunks[0] if clean_chunks else '')

    sec_document_inventory_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'primary_document_url': row.primary_document_url,
        'fetch_status': 'fetched',
        'raw_document_char_count': len(raw_text),
    })

    sec_document_text_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'primary_document_url': row.primary_document_url,
        'raw_text': raw_text,
        'clean_text': clean_text,
        'risk_factors_text': section_previews['risk_factors_preview'],
        'mda_text': section_previews['mda_preview'],
        'business_text': section_previews['business_preview'],
        'selected_clean_text': selected_text,
    })

    sec_cleaning_review_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'raw_document_char_count': len(raw_text),
        'clean_document_char_count': len(clean_text),
        'raw_chunk_count': len(raw_chunks),
        'clean_chunk_count': len(clean_chunks),
        'contains_risk_word': 'risk' in clean_text.lower(),
        'contains_management_word': 'management' in clean_text.lower(),
        'raw_text_preview': (raw_chunks[0][:800] if raw_chunks else ''),
        'clean_text_preview': (clean_chunks[0][:1200] if clean_chunks else ''),
    })

    sec_section_preview_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'risk_factors_preview': section_previews['risk_factors_preview'][:1200],
        'mda_preview': section_previews['mda_preview'][:1200],
        'business_preview': section_previews['business_preview'][:1200],
        'selected_clean_preview': selected_text[:1200],
    })

sec_document_inventory_df = pd.DataFrame(sec_document_inventory_rows)
sec_document_inventory_df.to_csv(REVIEW_DIR / 'sec_document_inventory_review.csv', index=False)

sec_document_text_full_df = pd.DataFrame(sec_document_text_rows)
sec_document_text_full_df.to_csv(REVIEW_DIR / 'sec_document_text_full.csv', index=False)

sec_cleaning_review_df = pd.DataFrame(sec_cleaning_review_rows)
sec_cleaning_review_df.to_csv(REVIEW_DIR / 'sec_cleaning_review.csv', index=False)

sec_section_preview_df = pd.DataFrame(sec_section_preview_rows)
sec_section_preview_df.to_csv(REVIEW_DIR / 'sec_section_preview_review.csv', index=False)

display(sec_document_inventory_df.head(10))
display(sec_cleaning_review_df[['ticker', 'form', 'raw_document_char_count', 'clean_document_char_count', 'clean_chunk_count', 'clean_text_preview']].head(10))
display(sec_section_preview_df[['ticker', 'form', 'risk_factors_preview', 'mda_preview', 'business_preview', 'selected_clean_preview']].head(10))
sec_document_text_full_df[['ticker', 'form', 'filing_date', 'selected_clean_text']].head(10)


## 5. FRED Macro Layer For Economic Weather

Fetch the macro series that help describe the broader economic backdrop.


In [ ]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')
fred_series = {
    'FEDFUNDS': 'Fed Funds Rate',
    'CPIAUCSL': 'CPI',
    'UNRATE': 'Unemployment Rate',
    'DGS10': '10Y Treasury',
    'DGS2': '2Y Treasury',
}

fred_frames = []
for series_id, title in fred_series.items():
    fred_url = (
        'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    )
    payload = fetch_json(fred_url)
    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]
    frame['series_id'] = series_id
    frame['title'] = title
    fred_frames.append(frame)

fred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()
fred_df.to_csv(EXTERNAL_DIR / 'fred' / 'macro_series_full_review.csv', index=False)
fred_df.tail(20)


## 6. FMP Company Profile Enrichment For Full Universe

Use the current stable FMP profile endpoint to enrich the full ticker universe with sector, industry, market cap, and instrument type hints.


In [ ]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')
profile_frames = []
for symbol in ticker_universe:
    fmp_url = f'https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={FMP_API_KEY}'
    profile_payload = fetch_json(fmp_url)
    profile_frames.append(pd.DataFrame(profile_payload if isinstance(profile_payload, list) else [profile_payload]))

fmp_profiles = pd.concat(profile_frames, ignore_index=True) if profile_frames else pd.DataFrame()
fmp_profiles.to_csv(EXTERNAL_DIR / 'fmp' / 'company_profiles_full_universe.csv', index=False)
display_columns = [col for col in ['symbol', 'companyName', 'sector', 'industry', 'marketCap', 'isEtf', 'isFund'] if col in fmp_profiles.columns]
fmp_profiles[display_columns].head(20)


## 7. GDELT News Metadata For Full Universe

Collect a manageable number of recent article candidates per ticker. This is still metadata-first; we will inflate only a smaller set of URLs in the next step.


In [ ]:
news_frames = []
for symbol in ticker_universe:
    gdelt_url = (
        'https://api.gdeltproject.org/api/v2/doc/doc'
        f'?query={quote(symbol)}&mode=artlist&maxrecords={MAX_GDELT_ARTICLES_PER_TICKER}&format=json'
    )
    payload = fetch_json(gdelt_url)
    articles = pd.DataFrame(payload.get('articles', []))
    if articles.empty:
        continue
    articles['ticker'] = symbol
    news_frames.append(articles)

gdelt_articles = pd.concat(news_frames, ignore_index=True) if news_frames else pd.DataFrame()
gdelt_articles.to_csv(EXTERNAL_DIR / 'gdelt' / 'articles_full_universe.csv', index=False)
gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']].head(20) if not gdelt_articles.empty else pd.DataFrame()


## 8. Inflate News URLs Into Full Text + Review

This section now keeps both a lightweight review table and the fuller extracted article text dataset for the inflated news URLs.


In [ ]:
article_reviews = []
article_text_rows = []
if not gdelt_articles.empty:
    unique_articles = (
        gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']]
        .dropna(subset=['url'])
        .drop_duplicates(subset=['url'])
        .head(MAX_NEWS_URLS_TO_INFLATE)
    )
    for row in unique_articles.itertuples(index=False):
        try:
            html = fetch_text(row.url)
            raw_text = html_to_text(html)
            clean_text = clean_article_text(raw_text)
            preview = simple_chunk(clean_text, chunk_size=1200)
            selected_text = preview[0] if preview else ''
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'fetch_status': 'fetched',
                'raw_char_count': len(raw_text),
                'clean_char_count': len(clean_text),
                'contains_market_words': any(word in clean_text.lower() for word in ['guidance', 'earnings', 'tariff', 'war', 'inflation', 'rate']),
                'text_preview': selected_text[:1200],
            })
            article_text_rows.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'raw_text': raw_text,
                'clean_text': clean_text,
                'selected_text': selected_text,
            })
        except Exception as error:
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'fetch_status': f'error: {type(error).__name__}',
                'raw_char_count': 0,
                'clean_char_count': 0,
                'contains_market_words': False,
                'text_preview': f'ERROR: {error}',
            })
            article_text_rows.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'raw_text': '',
                'clean_text': '',
                'selected_text': f'ERROR: {error}',
            })

article_review_df = pd.DataFrame(article_reviews)
article_review_df.to_csv(REVIEW_DIR / 'gdelt_article_review.csv', index=False)

article_text_full_df = pd.DataFrame(article_text_rows)
article_text_full_df.to_csv(REVIEW_DIR / 'gdelt_article_text_full.csv', index=False)

display(article_review_df.head(20) if not article_review_df.empty else pd.DataFrame())
article_text_full_df[['ticker', 'domain', 'title', 'selected_text']].head(20) if not article_text_full_df.empty else pd.DataFrame()


## 9. Source Usefulness Review

This final table turns the source-review exercise into a product-facing judgment: what each source is good for, where it is weak, and whether it belongs in the structured layer or the vector layer.


In [ ]:
source_review = pd.DataFrame([
    {
        'source_name': 'SEC filing metadata',
        'artifact': 'latest_reporting_filings',
        'storage_layer': 'structured',
        'best_for': 'Official filing inventory, dates, forms, and primary document URLs',
        'weakness': 'Not insight by itself; mostly pointers and metadata',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC company facts review',
        'artifact': 'company_facts_review',
        'storage_layer': 'structured',
        'best_for': 'Availability check and breadth of official company facts',
        'weakness': 'Summary only, not the full fact history',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC company facts full dataset',
        'artifact': 'company_facts_full_df',
        'storage_layer': 'structured',
        'best_for': 'Official fact history by tag, unit, filing date, and fiscal period',
        'weakness': 'Messy taxonomy and many rows per company; needs modeling',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC document inventory review',
        'artifact': 'sec_document_inventory_df',
        'storage_layer': 'structured',
        'best_for': 'Understanding which filing document was fetched and from where',
        'weakness': 'Still document metadata, not narrative insight',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC document text full dataset',
        'artifact': 'sec_document_text_full_df',
        'storage_layer': 'vector',
        'best_for': 'Reviewing full cleaned filing text and extracted narrative sections',
        'weakness': 'Limited to the inflated filing sample, not every filing in the universe',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC cleaning review',
        'artifact': 'sec_cleaning_review_df',
        'storage_layer': 'structured',
        'best_for': 'Judging whether filing text becomes readable after cleanup',
        'weakness': 'Does not yet isolate the best narrative section by itself',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC section preview review',
        'artifact': 'sec_section_preview_df',
        'storage_layer': 'vector',
        'best_for': 'Risk factors, business narrative, MD&A, and explainability review',
        'weakness': 'Section extraction is heuristic and may still miss some filings',
        'product_value': 'Very high',
    },
    {
        'source_name': 'FRED macro data',
        'artifact': 'fred_df',
        'storage_layer': 'structured',
        'best_for': 'Economic weather and macro regime context',
        'weakness': 'Not company-specific',
        'product_value': 'High',
    },
    {
        'source_name': 'FMP profiles',
        'artifact': 'fmp_profiles',
        'storage_layer': 'structured',
        'best_for': 'Sector, industry, market cap, and instrument metadata',
        'weakness': 'Convenience enrichment, not source-of-truth',
        'product_value': 'Medium to high',
    },
    {
        'source_name': 'GDELT article metadata',
        'artifact': 'gdelt_articles',
        'storage_layer': 'structured',
        'best_for': 'Narrative discovery and URL inventory',
        'weakness': 'Metadata alone is shallow and noisy',
        'product_value': 'Medium',
    },
    {
        'source_name': 'Inflated news article review',
        'artifact': 'article_review_df',
        'storage_layer': 'structured',
        'best_for': 'Quick quality check for fetched article URLs and text usefulness',
        'weakness': 'Preview-only summary of the inflated article sample',
        'product_value': 'Medium',
    },
    {
        'source_name': 'Inflated news article text full dataset',
        'artifact': 'article_text_full_df',
        'storage_layer': 'vector',
        'best_for': 'Reviewing cleaned article text for themes, event risk, and market context',
        'weakness': 'Limited to the inflated article sample and source HTML quality',
        'product_value': 'Medium to high',
    },
])
source_review.to_csv(REVIEW_DIR / 'source_usefulness_review.csv', index=False)
source_review